Graph Streaming :- LangGraph graph streaming is a core capability that lets you incrementally observe and output data incrementally during AI workflow execution, rather than waiting for the entire process to finish. It yields partial results, allowing developers to build responsive, real-time interfaces and applications

In [ ]:
"""
LangGraph Stream vs Astream Comparison:

| Feature          | graph.stream()                        | graph.astream()                          |
| :--------------- | :------------------------------------ | :--------------------------------------- |
| Execution Type   | Synchronous (Blocking)                | Asynchronous (Non-blocking)              |
| Loop Syntax      | Standard `for chunk in ...`           | Async `async for chunk in ...`           |
| Context          | Scripts, CLI tools, basic notebooks   | Production web apps, APIs, WebSockets    |
| Main Thread      | Blocks the thread while waiting       | Frees the thread to handle other tasks   |
| Required Env     | Standard Python functions             | Must be called inside `async def`        |
"""

In [ ]:
"""
LangGraph Stream Modes:
Pass one or more of these modes as a list to `.stream()` or `.astream()`.

| Mode        | Type                   | Description                                                                     |
| :---------- | :--------------------- | :------------------------------------------------------------------------------ |
| values      | ValuesStreamPart       | Full state after each step.                                                     |
| updates     | UpdatesStreamPart      | State updates after each step. Multi-updates in one step stream separately.    |
| messages    | MessagesStreamPart     | 2-tuples of (LLM token, metadata) from LLM calls.                               |
| custom      | CustomStreamPart       | Custom data emitted from nodes via `get_stream_writer`.                         |
| checkpoints | CheckpointStreamPart   | Checkpoint events (same format as `get_state()`). Requires a checkpointer.      |
| tasks       | TasksStreamPart        | Task start/finish events with results and errors. Requires a checkpointer.      |
| debug       | DebugStreamPart        | All available info — combines checkpoints and tasks with extra metadata.        |
"""

In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    topic: str
    joke: str
    end: str

def generate_joke(state: State):
    return {"joke": f"Why did the {state['topic']} go to school? To get a sundae education!"}

def ending(state: State):
     return {"end": "Graph ended successfully"}

graph = (
    StateGraph(State)
    .add_node(generate_joke)
    .add_node(ending)
    .add_edge(START, "generate_joke")
    .add_edge("generate_joke", "ending")
    .add_edge("ending" , END)
    .compile()
)

In [ ]:
## If we use stream mode as updates it will return State updates after each step. Multiple updates in the same step are streamed separately.
for chunk in graph.stream({"topic": "ice cream"}, stream_mode="updates", version="v2",):
        print(chunk)

{'type': 'updates', 'ns': (), 'data': {'generate_joke': {'joke': 'Why did the ice cream go to school? To get a sundae education!'}}}
{'type': 'updates', 'ns': (), 'data': {'ending': {'end': 'Graph ended successfully'}}}


In [ ]:
## the only output differences in verson v1 & version v2
for chunk in graph.stream({"topic": "ice cream"}, stream_mode="updates", version="v1",):
        print(chunk)

{'generate_joke': {'joke': 'Why did the ice cream go to school? To get a sundae education!'}}
{'ending': {'end': 'Graph ended successfully'}}


In [ ]:
## If we use stream nide as values it will return Full state after each step.
for chunk in graph.stream({"topic": "ice cream"}, stream_mode="values", version="v2",):
    print(chunk)

{'type': 'values', 'ns': (), 'data': {'topic': 'ice cream'}, 'interrupts': ()}
{'type': 'values', 'ns': (), 'data': {'topic': 'ice cream', 'joke': 'Why did the ice cream go to school? To get a sundae education!'}, 'interrupts': ()}
{'type': 'values', 'ns': (), 'data': {'topic': 'ice cream', 'joke': 'Why did the ice cream go to school? To get a sundae education!', 'end': 'Graph ended successfully'}, 'interrupts': ()}


In [9]:
## the only output differences in verson v1 & version v2
for chunk in graph.stream({"topic": "ice cream"}, stream_mode="values", version="v1",):
    print(chunk)

{'topic': 'ice cream'}
{'topic': 'ice cream', 'joke': 'Why did the ice cream go to school? To get a sundae education!'}
{'topic': 'ice cream', 'joke': 'Why did the ice cream go to school? To get a sundae education!', 'end': 'Graph ended successfully'}


In [13]:
## If we use stream mode as updates it will return State updates after each step. Multiple updates in the same step are streamed separately.
async for chunk in graph.astream({"topic": "ice cream"}, stream_mode="updates", version="v2",):
        print(chunk)

{'type': 'updates', 'ns': (), 'data': {'generate_joke': {'joke': 'Why did the ice cream go to school? To get a sundae education!'}}}
{'type': 'updates', 'ns': (), 'data': {'ending': {'end': 'Graph ended successfully'}}}
